In [28]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset

In [3]:
# ── 1. Load dataset ──────────────────────────────────────────────────────────
raw = load_dataset("roneneldan/TinyStories")
texts = raw["train"]["text"][:50_000]

# ── 2. Build character-level vocabulary ──────────────────────────────────────
corpus = "\n".join(texts)
chars  = sorted(set(corpus))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join(itos[i] for i in l)

print(f"Vocab size: {vocab_size}")

# ── 3. Encode corpus ─────────────────────────────────────────────────────────
data = torch.tensor(encode(corpus), dtype=torch.long)
n    = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Vocab size: 103


## Models
### 1. Bigram Language Model
### 2. MLP Language Model

In [29]:
# ── 4. Models ─────────────────────────────────────────────────────────────────

class BigramLM(nn.Module):
    def __init__(self, vocab_size: int):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding(idx)          # (B, T, vocab_size)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            probs     = F.softmax(logits[:, -1, :], dim=-1)
            idx_next  = torch.multinomial(probs, num_samples=1)
            idx       = torch.cat([idx, idx_next], dim=1)
        return idx


class MLPLM(nn.Module):
    def __init__(self, vocab_size: int, block_size: int, emb_dim: int = 64, hidden: int = 256):
        super().__init__()
        self.block_size = block_size
        self.embedding  = nn.Embedding(vocab_size, emb_dim)
        self.net = nn.Sequential(
            nn.Linear(block_size * emb_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, vocab_size),
        )

    def forward(self, idx, targets=None):
        # Always take the last block_size tokens — fixes the shape mismatch during generate
        idx    = idx[:, -self.block_size:]              # (B, block_size)
        B      = idx.size(0)
        x      = self.embedding(idx)                    # (B, block_size, emb_dim)
        x      = x.view(B, -1)                         # (B, block_size * emb_dim)
        logits = self.net(x).unsqueeze(1)               # (B, 1, vocab_size)

        loss = None
        if targets is not None:
            target_last = targets[:, -1]                # predict the next token after context
            loss = F.cross_entropy(logits.squeeze(1), target_last)

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # Crop before passing in — forward also crops but this keeps things explicit
            idx_crop = idx[:, -self.block_size:]
            logits, _ = self(idx_crop)
            probs     = F.softmax(logits[:, -1, :], dim=-1)
            idx_next  = torch.multinomial(probs, num_samples=1)
            idx       = torch.cat([idx, idx_next], dim=1)
        return idx

After implementing the Bigram and MLP model, we realised it still lacked depth. Further research in Attention Mechanism is required.


### 3. Attention Model
We verified the causal mask by applying -inf to the upper triangle of the attention scores before softmax, which converts those positions to 0. This enforces the autoregressive property where each token can only attend to itself and past tokens, never future ones.

In [30]:
# In Class Head
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False) 
value = nn.Linear(C, head_size, bias=False)

k = (key(x))   # (B, T, head_size)
q = (query(x)) # (B, T, head_size)
v = value(x)  # (B, T, head_size) 

wei = q @ k.transpose(-2, -1) # (B, 16, T) -> (B, T, T)

tril = torch.tril(torch.ones(T, T)) # (T, T)
wei = wei.masked_fill(tril == 0, float("-inf"))


In [31]:
wei[0]

tensor([[ 3.2820,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 1.5432,  2.6198,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.9959,  0.5879, -0.0165,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.3058, -0.8083, -0.2820, -1.3100,    -inf,    -inf,    -inf,    -inf],
        [-1.0029,  0.8299,  0.2366,  0.0856, -0.0997,    -inf,    -inf,    -inf],
        [ 2.4852, -0.4636,  0.3082, -0.2299, -0.0853, -0.9965,    -inf,    -inf],
        [-0.2601,  0.1520, -0.3442, -0.4045,  0.7137, -0.7011, -1.5921,    -inf],
        [-0.5284, -0.6851, -0.7585, -0.8548,  0.3287, -0.6963,  0.5581, -1.5777]],
       grad_fn=<SelectBackward0>)

In [32]:
wei = F.softmax(wei, dim=-1) # (B, T, T)
out = wei @ v

out.shape

torch.Size([4, 8, 16])

In [33]:
wei[0]


tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2542, 0.7458, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1171, 0.5709, 0.3119, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3338, 0.2020, 0.3419, 0.1223, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0619, 0.3872, 0.2140, 0.1840, 0.1529, 0.0000, 0.0000, 0.0000],
        [0.7467, 0.0391, 0.0847, 0.0494, 0.0571, 0.0230, 0.0000, 0.0000],
        [0.1274, 0.1924, 0.1171, 0.1103, 0.3373, 0.0820, 0.0336, 0.0000],
        [0.1011, 0.0865, 0.0804, 0.0730, 0.2383, 0.0855, 0.2998, 0.0354]],
       grad_fn=<SelectBackward0>)

In [34]:
# ── Attention Building Blocks ─────────────────────────────────────────────────

class Head(nn.Module):
    """One head of causal self-attention."""
    def __init__(self, head_size, emb_dim, block_size, dropout=0.1):
        super().__init__()
        self.key   = nn.Linear(emb_dim, head_size, bias=False)
        self.query = nn.Linear(emb_dim, head_size, bias=False)
        self.value = nn.Linear(emb_dim, head_size, bias=False)
        # tril is not a parameter, so register as a buffer
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)    # (B, T, head_size)
        q = self.query(x)  # (B, T, head_size)
        v = self.value(x)  # (B, T, head_size)

        # Attention scores — scale by sqrt(head_size) to keep variance stable
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)  # (B, T, T)

        # Causal mask — tokens can only attend to past tokens (not the future)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        return wei @ v  # (B, T, head_size)


class MultiHeadAttention(nn.Module):
    """Several attention heads running in parallel, then projected back."""
    def __init__(self, num_heads, head_size, emb_dim, block_size, dropout=0.1):
        super().__init__()
        self.heads = nn.ModuleList([
            Head(head_size, emb_dim, block_size, dropout) for _ in range(num_heads)
        ])
        self.proj    = nn.Linear(num_heads * head_size, emb_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = torch.cat([h(x) for h in self.heads], dim=-1)  # concat head outputs
        return self.dropout(self.proj(x))


class FeedForward(nn.Module):
    """Per-token MLP applied after attention (think: 'process what you gathered')."""
    def __init__(self, emb_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),
            nn.ReLU(),
            nn.Linear(4 * emb_dim, emb_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, emb_dim, block_size, dropout=0.1):
        super().__init__()
        self.attn = Head(emb_dim, emb_dim, block_size, dropout)  # head_size = emb_dim
        self.ff   = FeedForward(emb_dim, dropout)
        self.ln1  = nn.LayerNorm(emb_dim)
        self.ln2  = nn.LayerNorm(emb_dim)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


# ── The full model ─────────────────────────────────────────────────────────────

class TransformerLM(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=128, num_heads=4, num_layers=4, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        # Two embeddings: WHAT the token is + WHERE it is
        self.token_emb    = nn.Embedding(vocab_size, emb_dim)
        self.position_emb = nn.Embedding(block_size, emb_dim)
        self.blocks = nn.Sequential(*[
            TransformerBlock(emb_dim, block_size, dropout)
            for _ in range(num_layers)
        ])
        self.ln_f  = nn.LayerNorm(emb_dim)          # final norm
        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_emb(idx)                          # (B, T, emb_dim)
        pos_emb = self.position_emb(torch.arange(T, device=idx.device))  # (T, emb_dim)
        x = tok_emb + pos_emb                                  # broadcast over B

        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)                               # (B, T, vocab_size)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_crop = idx[:, -self.block_size:]       # never exceed block_size
            logits, _ = self(idx_crop)
            probs     = F.softmax(logits[:, -1, :], dim=-1)
            idx_next  = torch.multinomial(probs, num_samples=1)
            idx       = torch.cat([idx, idx_next], dim=1)
        return idx

In [35]:
# ── 5. Training helpers ───────────────────────────────────────────────────────
BATCH_SIZE = 64
BLOCK_SIZE = 8
EVAL_ITERS = 200
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

def get_batch(split: str):
    src = train_data if split == "train" else val_data
    ix  = torch.randint(len(src) - BLOCK_SIZE, (BATCH_SIZE,))
    x   = torch.stack([src[i : i + BLOCK_SIZE]         for i in ix])
    y   = torch.stack([src[i + 1 : i + BLOCK_SIZE + 1] for i in ix])
    return x.to(DEVICE), y.to(DEVICE)

@torch.no_grad()
def estimate_loss(model):
    model.eval()
    out = {}
    for split in ("train", "val"):
        losses = torch.zeros(EVAL_ITERS)
        for k in range(EVAL_ITERS):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

In [36]:
# ── 6. Config — swap model here ───────────────────────────────────────────────
# model = BigramLM(vocab_size).to(DEVICE)

# model = MLPLM(vocab_size, block_size=BLOCK_SIZE).to(DEVICE)

BLOCK_SIZE = 64   # increase this — attention benefits from longer context
model = TransformerLM(
    vocab_size  = vocab_size,
    block_size  = BLOCK_SIZE,
    emb_dim     = 128,
    num_heads   = 4,
    num_layers  = 4,
    dropout     = 0.1,
).to(DEVICE)

LR = 3e-4
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

NUM_EPOCHS      = 1
steps_per_epoch = len(train_data) // (BATCH_SIZE * BLOCK_SIZE)
MAX_ITERS       = NUM_EPOCHS * steps_per_epoch
EVAL_EVERY      = 1_000

print(f"Training for {MAX_ITERS} steps (~{NUM_EPOCHS} epoch(s))")

# ── 7. Train ──────────────────────────────────────────────────────────────────
for step in range(MAX_ITERS):
    if step % EVAL_EVERY == 0:
        losses = estimate_loss(model)
        print(f"step {step:>6} | train loss {losses['train']:.4f} | val loss {losses['val']:.4f}")

    xb, yb = get_batch("train")
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


Training for 9801 steps (~1 epoch(s))
step      0 | train loss 4.7386 | val loss 4.7393
step   1000 | train loss 1.5539 | val loss 1.5548
step   2000 | train loss 1.3379 | val loss 1.3481
step   3000 | train loss 1.2419 | val loss 1.2553
step   4000 | train loss 1.1863 | val loss 1.1965
step   5000 | train loss 1.1487 | val loss 1.1670
step   6000 | train loss 1.1194 | val loss 1.1254
step   7000 | train loss 1.1009 | val loss 1.1105
step   8000 | train loss 1.0850 | val loss 1.0893
step   9000 | train loss 1.0671 | val loss 1.0752


In [37]:
import json
import os

SAVE_DIR = "./runs/progression_bigram"
os.makedirs(SAVE_DIR, exist_ok=True)

torch.save(model.state_dict(), os.path.join(SAVE_DIR, "model.pt"))

config = {
    "model_type": model.__class__.__name__,
    "vocab_size": vocab_size,
    "block_size": BLOCK_SIZE,
    "batch_size": BATCH_SIZE,
    "device": DEVICE,
}

with open(os.path.join(SAVE_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

with open(os.path.join(SAVE_DIR, "stoi.json"), "w") as f:
    json.dump(stoi, f, indent=2)

print(f"Saved model to {SAVE_DIR}")


Saved model to ./runs/progression_bigram


In [38]:
# ── 8. Generate ───────────────────────────────────────────────────────────────
def generate_from_prompt(model, prompt: str, max_new_tokens: int = 500) -> str:
    model.eval()
    
    # Encode prompt — handle unknown chars gracefully
    encoded = [stoi[c] for c in prompt if c in stoi]
    if not encoded:
        print("Warning: no recognisable characters in prompt, starting from scratch.")
        encoded = [0]
    
    context = torch.tensor(encoded, dtype=torch.long, device=DEVICE).unsqueeze(0)  # (1, T)
    generated = model.generate(context, max_new_tokens=max_new_tokens)[0].tolist()
    return decode(generated)


prompt = "Once upon a time, there was a little girl"
print("\n── Generated text ──")
print(generate_from_prompt(model, prompt, max_new_tokens=500))


── Generated text ──
Once upon a time, there was a little girl kind. And they had an envelight things it wash and if it looked her and reful on had a big, for the park, enjoyed for a punial, this pocket and cry. The penny beach they were happy because his fridges.

"I want?"

Leople they played in them. Lily said, "Yes, let's please!" Then gazed and starting, Lily. You are mighty and he asked her and Spot. Yes, sorry, memonaged to getting clouds and were brother back to eat. Then it sappeared to go on to the paont and do.

He couldn't find her mom and dang
